# OBS Gradient Artifact Removal Demo

Este notebook carga un registro simultaneo EEG-fMRI (`fmrirestingec`), excluye el ultimo canal de ECG del pipeline, visualiza la senal antes de la limpieza, aplica OBS sin triggers y muestra comparaciones antes y despues.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np

START_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (START_DIRECTORY, *START_DIRECTORY.parents) if (path / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root containing src/.")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from functions.obs_ga import run_obs_pipeline

In [2]:
DEFAULT_EEG_ROOT = Path(
    r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG"
)
SUBJECT = "sub-007"
TASK = "fmrirestingec"
TR = 2.0
OFFSET = 0
REFERENCE_CHANNEL = 9
EXCLUDE_LAST_CHANNEL = True
N_COMPONENTS = 4
WINDOW_SIZE = 21
MAX_LAG = 0
REMOVE_MEAN = True
NORMALIZE = False
DISPLAY_SECONDS = 500

In [3]:
def get_eeg_set_path(subject: str, task: str = TASK, eeg_root: Path = DEFAULT_EEG_ROOT) -> Path:
    eeg_path = eeg_root / subject / "eeg" / f"{subject}_task-{task}_eeg.set"
    if not eeg_path.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_path}")
    return eeg_path


def load_raw_eeg(eeg_path: Path) -> mne.io.BaseRaw:
    return mne.io.read_raw_eeglab(eeg_path, preload=True, verbose="ERROR")


def prepare_raw_for_obs(raw: mne.io.BaseRaw, exclude_last_channel: bool = True) -> mne.io.BaseRaw:
    if exclude_last_channel:
        return raw.copy().pick(raw.ch_names[:-1])
    return raw.copy()


def plot_channel_before_after(
    raw_before: mne.io.BaseRaw,
    cleaned_data: np.ndarray,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    n_samples = min(int(round(duration_s * fs)), raw_before.n_times, cleaned_data.shape[1])
    time = np.arange(n_samples) / fs

    before = raw_before.get_data(picks=[channel_index])[0, :n_samples]
    after = cleaned_data[channel_index, :n_samples]
    channel_name = raw_before.ch_names[channel_index]

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(time, before, linewidth=0.8)
    axes[0].set_title(f"Before OBS | {channel_name}")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time, after, linewidth=0.8, color="tab:orange")
    axes[1].set_title(f"After OBS | {channel_name}")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_overlay_before_after(
    raw_before: mne.io.BaseRaw,
    cleaned_data: np.ndarray,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    n_samples = min(int(round(duration_s * fs)), raw_before.n_times, cleaned_data.shape[1])
    time = np.arange(n_samples) / fs
    before = raw_before.get_data(picks=[channel_index])[0, :n_samples]
    after = cleaned_data[channel_index, :n_samples]
    channel_name = raw_before.ch_names[channel_index]

    plt.figure(figsize=(14, 4))
    plt.plot(time, before, label="Before OBS", linewidth=0.8, alpha=0.75)
    plt.plot(time, after, label="After OBS", linewidth=0.8, alpha=0.75)
    plt.title(f"Overlay Before/After | {channel_name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_segment_basis_comparison(result: dict, channel_name: str, reference_channel: int = 0) -> None:
    raw_segment = result["segmented_signal"][reference_channel, 0]
    aligned_segment = result["aligned_segments"][reference_channel, 0]
    cleaned_segment = result["cleaned_segments"][reference_channel, 0]
    basis = result["basis"]

    if basis.ndim == 3:
        basis_to_plot = basis[0]
    else:
        basis_to_plot = basis

    active = np.any(np.abs(basis_to_plot) > 0, axis=1)
    basis_to_plot = basis_to_plot[active]
    samples = np.arange(raw_segment.shape[0])

    plt.figure(figsize=(14, 5))
    plt.plot(samples, raw_segment, label="Raw segment", linewidth=0.8, alpha=0.8)
    plt.plot(samples, aligned_segment, label="Aligned segment", linewidth=0.8, alpha=0.8)
    plt.plot(samples, cleaned_segment, label="Cleaned segment", linewidth=0.8, alpha=0.8)
    for idx, component in enumerate(basis_to_plot[: min(3, basis_to_plot.shape[0])], start=1):
        plt.plot(samples, component, label=f"Basis {idx}", linewidth=1.0)
    plt.title(f"Segment and OBS Basis Comparison | {channel_name}")
    plt.xlabel("Samples within TR")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [4]:
eeg_path = get_eeg_set_path(SUBJECT)
raw = load_raw_eeg(eeg_path)
raw_eeg = prepare_raw_for_obs(raw, exclude_last_channel=EXCLUDE_LAST_CHANNEL)
eeg_data = raw_eeg.get_data()
fs = float(raw_eeg.info["sfreq"])

print(f"Subject: {SUBJECT}")
print(f"Task: {TASK}")
print(f"EEG path: {eeg_path}")
print(f"Original shape: {raw.get_data().shape}")
print(f"OBS input shape: {eeg_data.shape}")
print(f"Sampling frequency: {fs} Hz")
print(f"Reference channel: {REFERENCE_CHANNEL} ({raw_eeg.ch_names[REFERENCE_CHANNEL]})")
if EXCLUDE_LAST_CHANNEL:
    print(f"Excluded channel: {raw.ch_names[-1]}")

Subject: sub-007
Task: fmrirestingec
EEG path: C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-007\eeg\sub-007_task-fmrirestingec_eeg.set
Original shape: (33, 628889)
OBS input shape: (32, 628889)
Sampling frequency: 1000.0 Hz
Reference channel: 9 (E10)
Excluded channel: ECG


## Visualizacion antes de limpiar

In [5]:
mne.viz.set_browser_backend("qt")
raw_eeg.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

Using qt as 2D backend.
Channels marked as bad:
none


## Aplicacion del pipeline OBS

In [9]:
result = run_obs_pipeline(
    signal=eeg_data,
    TR=TR,
    fs=fs,
    offset=OFFSET,
    reference_channel=REFERENCE_CHANNEL,
    n_components=N_COMPONENTS,
    max_lag=MAX_LAG,
    window_size=WINDOW_SIZE,
    remove_mean=REMOVE_MEAN,
    normalize=NORMALIZE,
)

cleaned_eeg = result["cleaned_signal"]

print(f"Samples per TR: {result['T_samples']}")
print(f"Offset used: {result['offset']} samples")
print(f"First 10 lags: {result['lags'][:10]}")
print(f"Cleaned EEG shape: {cleaned_eeg.shape}")

Samples per TR: 2000
Offset used: 0 samples
First 10 lags: [0 0 0 0 0 0 0 0 0 0]
Cleaned EEG shape: (32, 628889)


## Visualizacion despues de limpiar

In [7]:
raw_clean = raw_eeg.copy()
raw_clean._data = cleaned_eeg.copy()

In [8]:
raw_clean.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

Channels marked as bad:
none


In [ ]:
plot_channel_before_after(raw_eeg, cleaned_eeg, channel_index=REFERENCE_CHANNEL, duration_s=DISPLAY_SECONDS)
plot_overlay_before_after(raw_eeg, cleaned_eeg, channel_index=REFERENCE_CHANNEL, duration_s=DISPLAY_SECONDS)
plot_segment_basis_comparison(result, raw_eeg.ch_names[REFERENCE_CHANNEL], reference_channel=REFERENCE_CHANNEL)